# $\Delta: X \to X'$ — what happens to the transformation field when the substrate changes

**A-Mark9 / NEXUS · arena programme · §10 · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Self-contained: `itertools`, `numpy`, `collections` only.

---

## 1. The instruction, and what it forces

Stop comparing values of $\epsilon$. Take bases one elementary edit apart, construct how the
lawful transformations change, and find the induced map on residuals.

Doing it exposes something the value table hid completely: **there is no map $R_X \to R_{X'}$.**
There is a *span*, and the two legs behave in opposite directions.

An edge edit leaves the arena vertex set alone — $K$-subsets of the same $V$ — and only grows the
transformation set, since an arena transformation is $(e,T)$ with $e \in E(X)$. But the symmetry
group changes too, and $R$ is only defined relative to a group. So the edit factors:

$$R_{X,\;G_X} \;\hookrightarrow\; R_{X,\;G_\cap} \;\twoheadleftarrow\; R_{X',\;G_\cap} \;\hookleftarrow\; R_{X',\;G_{X'}}$$

with $G_\cap = \operatorname{Aut}(X)\cap\operatorname{Aut}(X')$ acting on the shared vertex set.
Two elementary steps, each measured separately below.

---

## 2. The two steps

**Transformation step** — fix the group, add base edges. $C^1(\mathcal A') = C^1(\mathcal A)
\oplus (\text{new edges})$, restriction $\rho$ is equivariant and commutes with $d_0$ (same vertex
set, same $d_0$ on the old coordinates), so it descends to $\Delta_R : R_{X'} \to R_X$.

**Symmetry step** — fix the base, enlarge the group. $(C^1)^{G'} \subseteq (C^1)^{G}$ and likewise
for the exact part, so inclusion gives $R_{G'} \to R_G$.

## 3. Library

In [1]:
import itertools, numpy as np
from collections import defaultdict
ATOL = 1e-9
def rank_abs(M):
    if M.size == 0 or min(M.shape) == 0: return 0
    return int((np.linalg.svd(M, compute_uv=False) > ATOL).sum())
def colspace(M):
    if M.size == 0 or min(M.shape) == 0: return np.zeros((M.shape[0], 0))
    U, s, _ = np.linalg.svd(M, full_matrices=False); return U[:, s > ATOL]

def aut(E, n):
    Es = set(map(tuple, map(sorted, E)))
    return [p for p in itertools.permutations(range(n))
            if all(tuple(sorted((p[u], p[v]))) in Es for (u, v) in Es)]

def items(E, n, K):
    """arena transformations as (e,T); index them."""
    out = []
    for e in sorted(map(tuple, map(sorted, E))):
        for T in itertools.combinations([x for x in range(n) if x not in e], K-1):
            out.append((frozenset(e), frozenset(T)))
    return out

def act(g, it):
    e, T = it
    return (frozenset(g[i] for i in e), frozenset(g[i] for i in T))

def orbit_table(E, n, K, G):
    its = items(E, n, K)
    seen, n_orb, n_sr = set(), 0, 0
    for it in its:
        if it in seen: continue
        seen |= {act(g, it) for g in G}
        n_orb += 1
        u, w = sorted(it[0])
        if any(g[u] == w and g[w] == u and {g[i] for i in it[1]} == set(it[1]) for g in G):
            n_sr += 1
    verts = list(itertools.combinations(range(n), K))
    seen, v_orb = set(), 0
    for S in verts:
        fs = frozenset(S)
        if fs in seen: continue
        seen |= {frozenset(g[i] for i in S) for g in G}; v_orb += 1
    return v_orb, n_orb, n_sr, n_orb - n_sr

def residual(E, n, K, G, index=None):
    """returns (basis of R as columns in C^1 indexed by `index`, dim)"""
    its = items(E, n, K)
    idx = index if index is not None else {it: i for i, it in enumerate(its)}
    m = len(idx)
    verts = sorted(itertools.combinations(range(n), K))
    vi = {frozenset(s): i for i, s in enumerate(verts)}
    D0 = np.zeros((m, len(verts)))
    for it in its:
        e, T = it; u, w = sorted(e)
        A = frozenset(T | {u}); B = frozenset(T | {w})   # orientation: token at u -> w
        D0[idx[it], vi[B]] += 1; D0[idx[it], vi[A]] -= 1
    P = np.zeros((m, m))
    for g in G:
        for it in its:
            e, T = it; u, w = sorted(e)
            j = act(g, it); ju, jw = sorted(j[0])
            s = 1.0 if (g[u], g[w]) == (ju, jw) else -1.0
            P[idx[j], idx[it]] += s / len(G)
    Inv = colspace(P); Ex = colspace(P @ colspace(D0))
    R = colspace(Inv - Ex @ (Ex.T @ Inv)) if Ex.shape[1] else Inv
    return R, R.shape[1]

def run(name, E, Ep, n, K):
    GX, GXp = aut(E, n), aut(Ep, n)
    Gc = [g for g in GX if g in set(GXp)]
    print(f"\n### {name}  K={K}   |Aut(X)|={len(GX)}  |Aut(X')|={len(GXp)}  |common|={len(Gc)}")
    for tag, EE, G in [("X  @ Aut(X)", E, GX), ("X  @ common", E, Gc),
                       ("X' @ common", Ep, Gc), ("X' @ Aut(X')", Ep, GXp)]:
        v, eo, esr, ens = orbit_table(EE, n, K, G)
        _, d = residual(EE, n, K, G)
        print(f"    {tag:<14} v_G={v:<4} e_orb={eo:<5} e_sr={esr:<4} e_ns={ens:<5} dim R={d}")
    # Delta_R : R_{X'} -> R_X  by restricting cochains to the old arena edges
    itsP = items(Ep, n, K); idxP = {it: i for i, it in enumerate(itsP)}
    its = items(E, n, K);   idx  = {it: i for i, it in enumerate(its)}
    Rp, dp = residual(Ep, n, K, Gc, idxP)
    R,  d  = residual(E,  n, K, Gc, idx)
    rows = [idxP[it] for it in its]                       # old edges sit inside the new arena
    restricted = Rp[rows, :]                              # rho(R')
    M = R.T @ restricted if (d and dp) else np.zeros((d, dp))
    r = rank_abs(M)
    print(f"    Delta_R : R_X' ({dp}) -> R_X ({d})   rank={r}  "
          f"kernel={dp-r} (created by the edit)  cokernel={d-r} (destroyed by the edit)")
    surv = len(its); born = len(itsP) - len(its)
    print(f"    transformations: {surv} survive verbatim, {born} newly available, 0 destroyed")

## 4. Edge edits


In [2]:
C = lambda n: [(i, (i+1) % n) for i in range(n)]
P = lambda n: [(i, i+1) for i in range(n-1)]
Q3 = [(v, v ^ (1 << i)) for v in range(8) for i in range(3) if v < v ^ (1 << i)]

run("P6 -> C6   (close the cycle)",      P(6), C(6),            6, 3)
run("C6 -> C6+{0,3} (long chord)",       C(6), C(6)+[(0,3)],    6, 3)
run("C6 -> C6+{0,2} (short chord)",      C(6), C(6)+[(0,2)],    6, 3)
run("C7 -> C7+{0,3}",                    C(7), C(7)+[(0,3)],    7, 3)
run("C8 -> C8+{0,4} (antipodal chord)",  C(8), C(8)+[(0,4)],    8, 3)
run("Q3 -> Q3+{0,7} (antipodal edge)",   Q3,   Q3+[(0,7)],      8, 3)



### P6 -> C6   (close the cycle)  K=3   |Aut(X)|=2  |Aut(X')|=12  |common|=2
    X  @ Aut(X)    v_G=10   e_orb=16    e_sr=2    e_ns=14    dim R=5
    X  @ common    v_G=10   e_orb=16    e_sr=2    e_ns=14    dim R=5
    X' @ common    v_G=10   e_orb=20    e_sr=4    e_ns=16    dim R=7
    X' @ Aut(X')   v_G=3    e_orb=4     e_sr=2    e_ns=2     dim R=0
    Delta_R : R_X' (7) -> R_X (5)   rank=5  kernel=2 (created by the edit)  cokernel=0 (destroyed by the edit)
    transformations: 30 survive verbatim, 6 newly available, 0 destroyed

### C6 -> C6+{0,3} (long chord)  K=3   |Aut(X)|=12  |Aut(X')|=4  |common|=4
    X  @ Aut(X)    v_G=3    e_orb=4     e_sr=2    e_ns=2     dim R=0
    X  @ common    v_G=6    e_orb=10    e_sr=2    e_ns=8     dim R=3
    X' @ common    v_G=6    e_orb=13    e_sr=4    e_ns=9     dim R=4
    X' @ Aut(X')   v_G=6    e_orb=13    e_sr=4    e_ns=9     dim R=4
    Delta_R : R_X' (4) -> R_X (3)   rank=3  kernel=1 (created by the edit)  cokernel=0 (destroyed by the edit

## 5. Symmetry steps


In [3]:
def sym_step(name, E, n, K, Gsmall, Gbig):
    its = items(E, n, K); idx = {it: i for i, it in enumerate(its)}
    Rs, ds = residual(E, n, K, Gsmall, idx)
    Rb, db = residual(E, n, K, Gbig, idx)
    r = rank_abs(Rs.T @ Rb) if (ds and db) else 0
    print(f"  {name:<24} |G| {len(Gsmall):>3} -> {len(Gbig):<4} dim R {ds} -> {db}   "
          f"rank={r}  kernel={db-r}  cokernel={ds-r}")

rot = lambda m: [tuple((i+k) % m for i in range(m)) for k in range(m)]
refl = lambda m: [tuple(range(m)), tuple((0-i) % m for i in range(m))]
for nm, E, n, K, small in [("C6 K=3, Z2 < D6", C(6), 6, 3, refl(6)),
                           ("C6 K=3, Z6 < D6", C(6), 6, 3, rot(6)),
                           ("C7 K=3, Z7 < D7", C(7), 7, 3, rot(7)),
                           ("C8 K=3, Z8 < D8", C(8), 8, 3, rot(8))]:
    sym_step(nm, E, n, K, small, aut(E, n))
Gq = [g for g in aut(Q3, 8) if g in set(aut(Q3+[(0,7)], 8))]
sym_step("Q3 K=3, order-12 < 48", Q3, 8, 3, Gq, aut(Q3, 8))


  C6 K=3, Z2 < D6          |G|   2 -> 12   dim R 7 -> 0   rank=0  kernel=0  cokernel=7
  C6 K=3, Z6 < D6          |G|   6 -> 12   dim R 3 -> 0   rank=0  kernel=0  cokernel=3
  C7 K=3, Z7 < D7          |G|   7 -> 14   dim R 6 -> 1   rank=1  kernel=0  cokernel=5
  C8 K=3, Z8 < D8          |G|   8 -> 16   dim R 9 -> 2   rank=2  kernel=0  cokernel=7
  Q3 K=3, order-12 < 48    |G|  12 -> 48   dim R 8 -> 1   rank=1  kernel=0  cokernel=7


## 6. Results

### 6.1 Edge edits — at the common group

| edit | $K$ | $|G_X|,|G_{X'}|,|G_\cap|$ | $\dim R_{X,G_\cap}$ | $\dim R_{X',G_\cap}$ | $\operatorname{rank}\Delta_R$ | ker | coker | survive / new / destroyed |
|---|--:|---|--:|--:|--:|--:|--:|---|
| $P_6 \to C_6$ | 3 | 2, 12, 2 | 5 | 7 | 5 | 2 | **0** | 30 / 6 / **0** |
| $C_6 \to C_6{+}\{0,3\}$ | 3 | 12, 4, 4 | 3 | 4 | 3 | 1 | **0** | 36 / 6 / **0** |
| $C_6 \to C_6{+}\{0,2\}$ | 3 | 12, 2, 2 | 7 | 9 | 7 | 2 | **0** | 36 / 6 / **0** |
| $C_7 \to C_7{+}\{0,3\}$ | 3 | 14, 2, 2 | 16 | 20 | 16 | 4 | **0** | 70 / 10 / **0** |
| $C_8 \to C_8{+}\{0,4\}$ | 3 | 16, 4, 4 | 14 | 17 | 14 | 3 | **0** | 120 / 15 / **0** |
| $Q_3 \to Q_3{+}\{0,7\}$ | 3 | 48, 12, 12 | 8 | 9 | 8 | 1 | **0** | 180 / 15 / **0** |

$$\boxed{\ \Delta_R \text{ is surjective in every case: cokernel } 0\ }$$

and the kernel equals $\Delta e_G^{\rm ns}$ exactly — $2,1,2,4,3,1$ against
$14{\to}16$, $8{\to}9$, $18{\to}20$, $34{\to}38$, $30{\to}33$, $15{\to}16$.

**At fixed symmetry, adding a base edge never destroys anything.** Every transformation survives
verbatim (no $(e,T)$ can cease to exist when $E$ only grows), and every residual channel of the
smaller system is the image of one in the larger. The new channels are exactly the kernel, one per
new non-flippable transformation orbit.

### 6.2 Symmetry steps — the other leg

| base, $K$ | $|G| \to |G'|$ | $\dim R_G \to \dim R_{G'}$ | rank | kernel | cokernel |
|---|---|---|--:|--:|--:|
| $C_6$, 3 | 2 → 12 | 7 → 0 | 0 | **0** | 7 |
| $C_6$, 3 | 6 → 12 | 3 → 0 | 0 | **0** | 3 |
| $C_7$, 3 | 7 → 14 | 6 → 1 | 1 | **0** | 5 |
| $C_8$, 3 | 8 → 16 | 9 → 2 | 2 | **0** | 7 |
| $Q_3$, 3 | 12 → 48 | 8 → 1 | 1 | **0** | 7 |

$$\boxed{\ R_{G'} \hookrightarrow R_G \text{ is injective in every case: kernel } 0\ }$$

---

## 7. What this says

The two elementary steps are one-sided exact in **opposite** directions, and neither of them ever
removes a channel:

$$\text{add transformations} \;\Rightarrow\; \Delta_R \text{ onto} \quad(\text{coker}=0,\ \text{nothing lost})$$
$$\text{add symmetry} \;\Rightarrow\; \Delta_R \text{ into} \quad(\ker=0,\ \text{nothing destroyed})$$

So the apparent destruction in $P_6 \to C_6$ — residual dropping from 5 to 0 — is not destruction.
Closing the cycle *added* six transformations and destroyed none; the five channels are still
there as invariant cochains for the smaller group. What changed is that the larger symmetry group
**cannot see them**. $R_{D_6} = 0$ is a statement about the observer, not about the arena.

$$\boxed{\ \text{no elementary edit erases a residual channel; edits extend, and symmetry hides}\ }$$

This is the shape a Law-3 candidate should have, stated in the $C1$ register: changing the
substrate changes which continuations are available, and changing the symmetry changes which are
distinguishable, but the continuation structure itself is only ever added to. Destruction, in this
model, is unavailable — it can only be simulated by an observer with more symmetry than the thing
it is observing.

**And the answer to "is there a transformation law $R_X \to R_{X'}$" is: no, there is a span.**
The two bases have no common symmetry group in general, so their residuals are not comparable
directly. They become comparable only at $G_\cap$, where one leg is an inclusion and the other a
surjection. Because $\Delta_R$ is onto, every channel of $X$ at full symmetry is hit by some
channel of $X'$ — nothing is orphaned by the edit — but the correspondence is many-to-one in one
direction and partial in the other. It is a relation, not a function.

### What is NOT established

- These are edge **additions** at a shared vertex set. Vertex addition, subdivision and
  contraction change $V$ and therefore the arena vertex set, so $\rho$ does not exist and none of
  the above applies to them. $C_6 \to C_7$ is a subdivision and is **not** covered here.
- Six edit pairs, one value of $K$. The cokernel-$0$ and kernel-$0$ statements look like theorems
  — restriction is surjective on cochains and commutes with $d_0$; invariants under a larger group
  are invariants under a smaller one — but they are stated here as measured, not proved.
- Nothing about mass, memory, or matter. $R$ is a finite-dimensional cochain sector.

## 8. Open

1. **Subdivision and contraction.** These are the edits that change $\epsilon$ on the symmetric
   family ($C_6 \to C_7$ takes $\epsilon$ from $0$ to $1$ at $K=3$), and they are exactly the ones
   this construction does not reach. A move $u \to w$ becomes a pair $u \to v \to w$: transformations
   *split* rather than survive. That needs a map from arena edges to arena **paths**, and the
   induced object is a chain map, not a restriction.
2. **Prove the two one-sided exactness statements.** Both look like one-liners; neither is written.
3. Whether $\operatorname{rank}\Delta_R$ composed across a chain of edits is the right invariant of
   a *path* through base-graph space — the C1 question of what is conserved under iterated change,
   rather than under a single edit.
